# Electron spin resonance of a spin-1 spin-1/2 ferrimagnet
This notebook simulates an ESR frequency sweep of the (1,1/2) ferrimagnet. It includes:
- Spin Hamiltonian
- ESR frequency sweeps
- Tipfield-dependent ESR 

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const

from spinfinity import ESR, SpinSys, bath, hamil
from spinfinity import helperfunctions as hf


%load_ext autoreload
%autoreload 2

## Spin Hamiltonian

We model the FePc–Fe(C6H6) complex using two coupled spins:

- Spin 0: Fe(C6H6), \(S_0 = 1\).
- Spin 1: FePc, \(S_1 = 1/2\).

The Hamiltonian includes isotropic antiferromagnetic exchange, axial
anisotropy on the spin-1 site, and an external magnetic field
$\mathbf{B} = B\hat{\mathbf{z}}$ acting on both spins:

$$
H =
J\,\mathbf{S}_0\cdot\mathbf{S}_1
+ D\,S_{0z}^{2}
- \mu_B B\left(g_0 S_{0z} + g_1 S_{1z}\right).
$$

Here, $g_0$ and $g_1$ are the respective g-factors. The Zeeman term
uses Spinfinity's negative-sign convention.

We use $J = 14\,\mathrm{meV}$ and $D = +1.9\,\mathrm{meV}$.

In [ ]:
# Define the Spin System
spin=SpinSys([1,0.5])
spin.gfactorvector = [2.0, 2.0]
spin.B = np.array([0.0, 0.0, 0.5])  # T; external magnetic field along z-axis

# Isotropic antiferromagnetic exchange
J_meV = 14.0
hamil.assign_ExchangeInteraction(
    spin,
    [(0, 1, (J_meV, J_meV, J_meV))],
)

# Axial anisotropy on Fe(C6H6) only
D_meV = 1.9
spin.Dvector = np.array([D_meV, 0.0])

hamil.build_Hamiltonian(spin)
hamil.calc_EigEnergies(spin)
hamil.calc_EigStates(spin)

eigenstate_result = hamil.plot_EigenMatrix(spin, plot_type="p")
energy_result = hamil.plot_EnergyVsS(
    spin,
    type="z",
    energyscale="meV",
)

## ESR frequency sweep on different sites

In [ ]:
# Tunneling parameters
spin.T = 0.5 # K
spin.V_DC = 10.0  # mV
spin.TipPolarization = [0, 0, 0.5]
spin.U = np.array([0.0, 0.0])
spin.G_st = 0.1e-9  # S
spin.G_ss = [1e-9, 1e-9]  # S
spin.G_tt = 1 # Allow for tip-tip scattering
spin.b0 = 0.0

bath.calc_Rates(spin)
bath.calc_Populations(spin, AllowPumping=True)
I_bg = bath.calc_TunnelCurrent(spin, AllowPumping=True)
print(f"Background tunneling current: {I_bg*1e12:.3e} pA")

# Parameters for the frequency sweep
T2 = 50.0  # ns; total coherence time which should fulfill T2 <= 2*T1
B1 = 2e-3  # T; microwave field amplitude 

freq_array = np.linspace(12, 16, 2000)  # GHz; frequency sweep range

spin.ReadoutSpin = 0  # index of the readout spin (Fe(C6H6) in this case)
# The calc function returns the frequency array, the current contrast, and the off-resonance current
_, DeltaI_Fe, I_off_Fe = ESR.calc_FreqSweep_ESRrate(
    spin,
    freq_array=freq_array,
    B1=B1,
    T2=T2,
    norm=False,
)

spin.ReadoutSpin = 1  # index of the readout spin (FePc in this case)
# The calc function returns the frequency array, the current contrast, and the off-resonance current
_, DeltaI_FePc, I_off_FePc = ESR.calc_FreqSweep_ESRrate(
    spin,
    freq_array=freq_array,
    B1=B1,
    T2=T2,
    norm=False,
)

# Restore readout spin to Fe(C6H6)
spin.ReadoutSpin = 0

# Plot the results
plt.figure(figsize=(8, 5))
plt.plot(freq_array, DeltaI_Fe*1e15, label="Fe(C6H6)", color="red")
plt.plot(freq_array, DeltaI_FePc*1e15, label="FePc", color="blue")
plt.xlabel("Frequency (GHz)")
plt.ylabel(f"$\Delta$ I (fA)")
plt.title("ESR spectra")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Tip-field dependence

In [ ]:
angle = 10.0 # degree
BTip_array = np.linspace(0.005, 0.08, 201)  # T
freq_array = np.linspace(13, 17, 2000)  # GHz
lw = 0.05 

spin.ReadoutSpin = 0  # index of the readout spin (Fe(C6H6) in this case)
return_dict = ESR.plot_Tipfieldmap(
    spin,
    freq_array=freq_array,
    BTip_array=BTip_array,
    lw=lw,
    norm=False,
    AllowPumping=False,
    Angle=angle,
)

spin.ReadoutSpin = 1  # index of the readout spin (FePc in this case)
return_dict = ESR.plot_Tipfieldmap(
    spin,
    freq_array=freq_array,
    BTip_array=BTip_array,
    lw=lw,
    norm=False,
    AllowPumping=False,
    Angle=angle,
)